[目录](./table_of_contents.ipynb)

# 卡尔曼滤波器数学

In [ ]:
%matplotlib inline

In [ ]:
#格式化书籍
import book_format
book_format.set_style()

如果您已经阅读到这里，我希望您认为卡尔曼滤波器的可怕声誉有些过度夸大了。当然，我对一些方程进行了概述，但我希望您的实现相当简单。其基本概念相当简单 - 取两个测量值或一个测量值和一个预测值，并选择输出在两者之间。如果您更相信测量值，您的猜测将更接近测量值，如果您认为预测值更准确，则您的猜测将更接近它。这不是火箭科学（开个小玩笑 - 正是这种数学使阿波罗登月并返回地球！）。

说实话，我一直在谨慎地选择我的问题。对于任意问题，设计卡尔曼滤波器矩阵可能会非常困难。尽管如此，我并没有太过于棘手。像牛顿运动方程这样的方程可以轻松地用于卡尔曼滤波器应用，并且它们占据了我们想要解决的问题的大部分。

我用代码和推理阐述了这些概念，而不是数学。但有些主题确实需要比我到目前为止使用的更多的数学。本章介绍了您将需要的其余内容的数学知识。

## 建模动态系统

*动态系统*是一种物理系统，其状态（位置、温度等）随时间演变。微积分是变化值的数学，因此我们使用微分方程来模拟动态系统。一些系统无法用微分方程建模，但在本书中我们不会遇到这些问题。

建模动态系统是几门大学课程的恰当主题。在某种程度上，没有什么可以替代几个学期的常微分方程和偏微分方程，再加上一个控制系统理论的研究生课程。如果你是一个业余爱好者，或者试图解决工作中一个非常具体的滤波问题，你可能没有时间和/或倾向去投入一年或更长时间的教育。

幸运的是，我可以呈现足够的理论，使我们能够为许多不同的卡尔曼滤波器创建系统方程。我的目标是让你达到一个阶段，你可以阅读一篇论文，并理解它的算法，然后实现它。背景数学很深奥，但在实践中，我们最终使用一些简单的技巧。

这是本书中最长的纯数学部分。你需要掌握这一部分的所有内容，才能理解扩展卡尔曼滤波器（EKF），这是最常见的非线性滤波器。我会涵盖更多现代滤波器，这些滤波器不需要这么多的数学知识。你现在可以选择略过，如果决定学习EKF可以回来再学习这一部分。

我们需要先了解卡尔曼滤波器使用的基本方程和假设。我们试图模拟现实世界的现象，那么我们需要考虑什么？

每个物理系统都有一个过程。例如，以某个速度行驶的汽车在固定的时间内行驶一定的距离，并且其速度随其加速度的函数而变化。我们用高中学过的著名的牛顿方程来描述这种行为。

$$
\begin{aligned}
v&=at\\
x &= \frac{1}{2}at^2 + v_0t + x_0
\end{aligned}
$$

学习微积分后，我们可以看到它们是这个形式：

$$ \mathbf v = \frac{d \mathbf x}{d t}, 
\quad \mathbf a = \frac{d \mathbf v}{d t} = \frac{d^2 \mathbf x}{d t^2}
$$

一个典型的汽车跟踪问题会让你计算在恒定速度或加速度下行驶的距离，就像我们在之前的章节中所做的那样。但是，我们知道这不是所有正在发生的事情。没有一辆汽车在完美的道路上行驶。道路上有颠簸、风阻和山丘会使速度上升和下降。悬挂系统是一个具有摩擦和不完美弹簧的机械系统。

除了最简单的问题外，完美地对系统进行建模是不可能的。我们被迫简化问题。在任何时间 $t$，我们都说真实状态（例如汽车的位置）是从不完美模型的预测值中加上一些未知的*过程噪声*：

$$
x(t) = x_{pred}(t) + noise(t)
$$

这并不意味着 $noise(t)$ 是一个我们可以通过分析得出的函数。这只是一个事实陈述 - 我们总是可以将真实值描述为预测值加上过程噪声。 "噪声" 并不意味着随机事件。如果我们在大气中跟踪一个抛出的球，并且我们的模型假设球在真空中，则空气阻力的影响在这种情况下是过程噪声。

在下一节中，我们将学习将一组高阶微分方程转换为一组一阶微分方程的技术。转换后的无噪声系统模型为：

$$\dot{\mathbf x} = \mathbf{Ax}$$

$\mathbf A$ 被称为 *系统动态矩阵*，因为它描述了系统的动态。现在我们需要对噪声进行建模。我们将其称为 $\mathbf w$，并将其添加到方程中。

$$\dot{\mathbf x} = \mathbf{Ax} + \mathbf w$$

$\mathbf w$ 可能不是一个好的选择作为变量名，但是您很快就会发现 Kalman 滤波器假设有 *白噪声*。

最后，我们需要考虑系统中的任何输入。我们假定有一个输入 $\mathbf u$，并且存在一个线性模型来定义该输入如何改变系统。例如，按下汽车油门会使其加速，重力会使球下落。两者都是控制输入。我们需要一个矩阵 $\mathbf B$ 将 $u$ 转换为对系统的影响。我们将其添加到方程中：

$$ \dot{\mathbf x} = \mathbf{Ax} + \mathbf{Bu} + \mathbf{w}$$

就是这样。这是 Kalman 博士设法解决的方程之一，如果我们假设 $\mathbf w$ 具有某些属性，则他找到了最佳的估计器。

## 动态系统的状态空间表示

我们推导出方程式

$$ \dot{\mathbf x} = \mathbf{Ax}+ \mathbf{Bu} + \mathbf{w}$$

然而，我们对 $\mathbf x$ 的导数不感兴趣，而是对 $\mathbf x$ 本身感兴趣。暂时忽略噪声，我们希望得到一个递归地找到时间 $t_k$ 时 $\mathbf x$ 值的方程，其中 $\mathbf x$ 在时间 $t_{k-1}$ 时的值为：

$$\mathbf x(t_k) = \mathbf F(\Delta t)\mathbf x(t_{k-1}) + \mathbf B(t_k)\mathbf u (t_k)$$

约定允许我们将 $\mathbf x(t_k)$ 写成 $\mathbf x_k$，这意味着 $\mathbf x$ 在 $t$ 的第 $k^{th}$ 个值的值。

$$\mathbf x_k = \mathbf{Fx}_{k-1} + \mathbf B_k\mathbf u_k$$

$\mathbf F$是众所周知的*状态转移矩阵*，因为它能够在离散时间步之间转换状态的值而得名。它与系统动力学矩阵$\mathbf A$非常相似。区别在于$\mathbf A$模拟了一组线性微分方程，是连续的。$\mathbf F$是离散的，并表示一组线性方程（而不是微分方程），它在离散时间步$\Delta t$内将$\mathbf x_{k-1}$转换到$\mathbf x_k$。

找到这个矩阵通常相当困难。方程$\dot x = v$是最简单的微分方程，我们可以简单地对其进行积分：

$$ \int\limits_{x_{k-1}}^{x_k}  \mathrm{d}x = \int\limits_{0}^{\Delta t} v\, \mathrm{d}t $$
$$x_k-x_{k-1} = v \Delta t$$
$$x_k = v \Delta t + x_{k-1}$$

这个方程是*递归*的：我们基于$k-1$时刻的值计算$k$时刻的$x$值。这种递归形式使我们能够以Kalman滤波器所需的形式表示系统（过程模型）：

$$\begin{aligned}
\mathbf x_k &= \mathbf{Fx}_{k-1}  \\
&= \begin{bmatrix} 1 & \Delta t \\ 0 & 1\end{bmatrix}
\begin{bmatrix}x_{k-1} \\ \dot x_{k-1}\end{bmatrix}
\end{aligned}$$

我们之所以能这样做，是因为$\dot x = v$是可能的最简微分方程。在物理系统中，几乎所有其他微分方程都会导致更复杂的微分方程，不能采用这种方法。

*状态空间*方法在阿波罗任务时期变得流行，主要归功于卡尔曼博士的工作。这个想法很简单。用一组$n$阶微分方程模拟一个系统。将它们转换成等价的一阶微分方程组。将它们放入前一节中使用的向量矩阵形式中：$\dot{\mathbf x} = \mathbf{Ax} + \mathbf{Bu}$。一旦处于这种形式，我们使用多种技术将这些线性微分方程转换为递推方程：

$$ \mathbf x_k = \mathbf{Fx}_{k-1} + \mathbf B_k\mathbf u_k$$

一些书籍将状态转移矩阵称为*基本矩阵*。许多人使用$\mathbf \Phi$而不是$\mathbf F$。基于控制理论的来源倾向于使用这些形式。

这些被称为*状态空间*方法，因为我们用系统状态来表示微分方程的解决方案。

### 从高阶方程组到一阶方程组的形式化


许多物理系统的模型需要带有控制输入$u$的二阶或更高阶微分方程：

$$a_n \frac{d^ny}{dt^n} + a_{n-1} \frac{d^{n-1}y}{dt^{n-1}} +  \dots + a_2 \frac{d^2y}{dt^2} + a_1 \frac{dy}{dt} + a_0 = u$$

状态空间方法需要一阶方程。通过为导数定义额外的变量然后求解，可以将任何高阶方程系统降低到一阶。

让我们举个例子。已知系统$\ddot{x} - 6\dot x + 9x = u$，找到等效的一阶方程式。我使用了点符号表示时间导数以增加清晰度。

第一步是将最高阶项隔离到等式的一侧。

$$\ddot{x} = 6\dot x - 9x + u$$

我们定义两个新变量：

$$\begin{aligned} x_1(t) &= x \\
x_2(t) &= \dot x
\end{aligned}$$

现在我们将它们代入原方程并求解。解得一组关于这些新变量的一阶方程。为了方便表示，惯例上省略$(t)$。

我们知道$\dot x_1 = x_2$和$\dot x_2 = \ddot{x}$。因此，

$$\begin{aligned}
\dot x_2 &= \ddot{x} \\
         &= 6\dot x - 9x + u\\
         &= 6x_2-9x_1 + u
\end{aligned}$$

因此，我们的一阶方程组为

$$\begin{aligned}\dot x_1 &= x_2 \\
\dot x_2 &= 6x_2-9x_1 + u\end{aligned}$$

如果您多加练习，就会熟练掌握它。隔离最高项，定义新变量及其导数，然后代入。

### 状态空间形式的一阶微分方程

将前一节中新定义的变量代入一阶方程中：

$$\frac{dx_1}{dt} = x_2,\,  
\frac{dx_2}{dt} = x_3, \, ..., \, 
\frac{dx_{n-1}}{dt} = x_n$$

则得到：


$$\frac{dx_n}{dt} = \frac{1}{a_n}\sum\limits_{i=0}^{n-1}a_ix_{i+1} + \frac{1}{a_n}u
$$

使用向量矩阵表示法，我们有：

$$\begin{bmatrix}\frac{dx_1}{dt} \\ \frac{dx_2}{dt} \\ \vdots \\ \frac{dx_n}{dt}\end{bmatrix} = 
\begin{bmatrix}\dot x_1 \\ \dot x_2 \\ \vdots \\ \dot x_n\end{bmatrix}=
\begin{bmatrix}0 & 1 & 0 &\cdots & 0 \\
0 & 0 & 1 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
-\frac{a_0}{a_n} & -\frac{a_1}{a_n} & -\frac{a_2}{a_n} & \cdots & -\frac{a_{n-1}}{a_n}\end{bmatrix}
\begin{bmatrix}x_1 \\ x_2 \\ \vdots \\ x_n\end{bmatrix} + 
\begin{bmatrix}0 \\ 0 \\ \vdots \\ \frac{1}{a_n}\end{bmatrix}u$$

然后我们将其写为 $\dot{\mathbf x} = \mathbf{Ax} + \mathbf{B}u$。

### 寻找时不变系统的基础矩阵

我们用以下状态空间形式表示系统方程：

$$ \dot{\mathbf x} = \mathbf{Ax}$$

其中$\mathbf A$是系统动态矩阵，我们需要找到*基本矩阵*$\mathbf F$，它将状态$\mathbf x$在时间间隔$\Delta t$内传播，方程为

$$\begin{aligned}
\mathbf x(t_k) = \mathbf F(\Delta t)\mathbf x(t_{k-1})\end{aligned}$$

换句话说，$\mathbf A$是一组连续的微分方程，我们需要$\mathbf F$成为一组离散的线性方程，以计算$\mathbf A$在离散时间步长上的变化。

通常，省略$t_k$和$(\Delta t)$，采用以下符号：

$$\mathbf x_k = \mathbf {Fx}_{k-1}$$

广义上讲，有三种常见的方法来为Kalman滤波器找到这个矩阵。最常用的技术是矩阵指数。线性时不变理论（也称为LTI系统理论）是第二种技术。最后是数值技术。您可能了解其他技术，但这三种是您在Kalman滤波器文献和实践中最有可能遇到的。

### 矩阵指数

方程 $\frac{dx}{dt} = kx$ 的解可以通过以下方式找到：

$$\begin{gathered}\frac{dx}{dt} = kx \\
\frac{dx}{x} = k\, dt \\
\int \frac{1}{x}\, dx = \int k\, dt \\
\log x = kt + c \\
x = e^{kt+c} \\
x = e^ce^{kt} \\
x = c_0e^{kt}\end{gathered}$$

当 $t=0$ 时，$x=x_0$。将这些代入上面的方程。

$$\begin{gathered}x_0 = c_0e^{k(0)} \\
x_0 = c_01 \\
x_0 = c_0 \\
x = x_0e^{kt}\end{gathered}$$

使用类似的数学方法，一阶方程

$$\dot{\mathbf x} = \mathbf{Ax} ,\, \, \, \mathbf x(0) = \mathbf x_0$$

的解为

$$\mathbf x = e^{\mathbf At}\mathbf x_0$$

代入 $F = e^{\mathbf At}$，我们可以写成

$$\mathbf x_k = \mathbf F\mathbf x_{k-1}$$

这正是我们要找的形式！我们已将找到基本矩阵的问题简化为找到 $e^{\mathbf At}$ 的值。

$e^{\mathbf At}$ 被称为[矩阵指数](https://en.wikipedia.org/wiki/Matrix_exponential)。它可以通过以下幂级数计算：

$$e^{\mathbf At} = \mathbf{I} + \mathbf{A}t  + \frac{(\mathbf{A}t)^2}{2!} + \frac{(\mathbf{A}t)^3}{3!} + ... $$

这个级数是通过对 $e^{\mathbf At}$ 进行泰勒级数展开得到的，这里不再赘述。

让我们用这个方法来找到牛顿运动方程的解。使用 $v$ 代替 $\dot x$，并假设速度恒定，我们得到线性矩阵-向量形式

$$\begin{bmatrix}\dot x \\ \dot v\end{bmatrix} =\begin{bmatrix}0&1\\0&0\end{bmatrix} \begin{bmatrix}x \\ v\end{bmatrix}$$

这是一个一阶微分方程，所以我们可以将 $\mathbf{A}=\begin{bmatrix}0&1\\0&0\end{bmatrix}$ 并解下面的方程。我用 $\Delta t$ 替换了 $t$ 来强调基本矩阵是离散的：

$$\mathbf F = e^{\mathbf A\Delta t} = \mathbf{I} + \mathbf A\Delta t  + \frac{(\mathbf A\Delta t)^2}{2!} + \frac{(\mathbf A\Delta t)^3}{3!} + ... $$

如果进行乘法，您将发现 $\mathbf{A}^2=\begin{bmatrix}0&0\\0&0\end{bmatrix}$，这意味着所有更高次幂的 $\mathbf{A}$ 也是 $\mathbf{0}$。因此，我们得到一个精确的答案，而不需要无限多的项：

$$
\begin{aligned}
\mathbf F &=\mathbf{I} + \mathbf A \Delta t + \mathbf{0} \\
&= \begin{bmatrix}1&0\\0&1\end{bmatrix} + \begin{bmatrix}0&1\\0&0\end{bmatrix}\Delta t\\
&= \begin{bmatrix}1&\Delta t\\0&1\end{bmatrix}
\end{aligned}$$

我们将其插入到 $\mathbf x_k= \mathbf{Fx}_{k-1}$ 中，得到

$$
\begin{aligned}
x_k &=\begin{bmatrix}1&\Delta t\\0&1\end{bmatrix}x_{k-1}
\end{aligned}$$

您将会认出这是我们在 **多元卡尔曼滤波器** 章节中解析推导的常速度卡尔曼滤波器的矩阵。

SciPy 的 linalg 模块包含一个 `expm()` 方法来计算矩阵指数。它不使用泰勒级数法，而是使用 [Padé 近似](https://en.wikipedia.org/wiki/Pad%C3%A9_approximant) 方法。计算矩阵指数的方法有许多种（至少19种），并且所有方法都会遇到数值困难[1]。当 $\mathbf A$ 很大时，您应该意识到这些问题。如果您搜索“pade approximation matrix exponential”，您会发现许多出版物专门研究此问题。

实际上，这可能对您并不重要，因为对于卡尔曼滤波器，我们通常只取泰勒级数的前两项。但是不要假设我处理问题的方式已经完整，然后就离开并尝试在没有对此技术的性能进行数值分析的情况下将此技术用于其他问题。有趣的是，解决$e^{\mathbf At}$的最喜欢方法之一是使用广义ode求解器。换句话说，他们做的与我们相反 - 将 $\mathbf A$ 转化为一组微分方程，然后使用数值技术来解决这个集合！

这是使用 `expm()`解决 $e^{\mathbf At}$的示例。

In [ ]:
import numpy as np
from scipy.linalg import expm

dt = 0.1
A = np.array([[0, 1], 
              [0, 0]])
expm(A*dt)

### 时间不变性

如果系统的行为取决于时间，我们可以说动态系统由一阶微分方程描述：

$$ g(t) = \dot x$$

然而，如果系统是*时不变的*，方程的形式为：

$$ f(x) = \dot x$$

什么是*时不变的*？想象一下家用音响。如果在时间$t$将信号$x$输入其中，它将输出一些信号$f(x)$。如果您改为在时间$t+\Delta t$执行输入，则输出信号仍将是相同的$f(x)$，只是时间发生了位移。

一个反例是$x(t)=\sin(t)$，其中系统$f(x)=t\,x(t)=t\sin(t)$。这不是时不变的；由于乘以t，不同时间的值将不同。飞机不是时不变的。如果您在以后的时间对飞机进行控制输入，则其行为将不同，因为它已经燃烧燃料，因此失去了重量。较低的重量会导致不同的行为。

我们可以通过对每一边积分来解决这些方程。我之前演示了如何积分时不变系统 $v = \dot x$。但是，积分时不变方程 $\dot x = f(x)$ 并不那么直接。使用*变量分离*技术，我们除以 $f(x)$ 并将 $dt$ 项移到右边，以便我们可以对每一边进行积分：

$$\begin{gathered}
\frac{dx}{dt} = f(x) \\
\int^x_{x_0} \frac{1}{f(x)} dx = \int^t_{t_0} dt
\end{gathered}$$

如果我们令 $F(x) = \int \frac{1}{f(x)} dx$，我们就得到了

$$F(x) - F(x_0) = t-t_0$$

然后我们用以下公式求解 $x$

$$\begin{gathered}
F(x) = t - t_0 + F(x_0) \\
x = F^{-1}[t-t_0 + F(x_0)]
\end{gathered}$$

换句话说，我们需要找到 $F$ 的反函数。这并不容易，而在STEM教育中，有相当多的课程是专门用来寻找这个问题的棘手且分析解决方案的。

然而，它们只是一些技巧，许多简单形式的 $f(x)$ 要么没有闭合形式的解决方案，要么存在极大的困难。因此，实践工程师会转向状态空间方法来找到近似解。

矩阵指数的优点在于我们可以将其用于任何任意的*时不变*微分方程组。然而，即使方程组不是时不变的，我们通常也会使用这种技术。例如，当飞机飞行时会燃烧燃料并且失去重量。但是，在一秒钟内失重的重量微不足道，因此该系统在那个时间步长上几乎是线性的。只要时间步长很短，我们的答案仍然会比较准确。

#### 示例：质量-弹簧-阻尼器模型

假设我们想要跟踪弹簧上的重物和连接到阻尼器上的运动，例如汽车的悬架。在某些输入 $u$ 下，运动的方程式为 $m$ 为质量，$k$ 为弹簧常数，$c$ 为阻尼力。

$$m\frac{d^2x}{dt^2} + c\frac{dx}{dt} +kx = u$$

为了符号上的方便，我将其写为：

$$m\ddot x + c\dot x + kx = u$$

我可以通过设置 $x_1(t)=x(t)$ 将其转换为一阶方程组，然后进行以下替换：

$$\begin{aligned}
x_1 &= x \\
x_2 &= \dot x_1 \\
\dot x_2 &= \ddot x_1 = \ddot x
\end{aligned}$$

出于常见习惯，我在符号上省略了 $(t)$。这给出方程：

$$m\dot x_2 + c x_2 +kx_1 = u$$

解出 $\dot x_2$ 我们得到一个一阶方程：

$$\dot x_2 = -\frac{c}{m}x_2 - \frac{k}{m}x_1 + \frac{1}{m}u$$

我们将此转换为矩阵形式：

$$\begin{bmatrix} \dot x_1 \\ \dot x_2 \end{bmatrix} = 
\begin{bmatrix}0 & 1 \\ -k/m & -c/m \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix} + 
\begin{bmatrix} 0 \\ 1/m \end{bmatrix}u$$

现在我们使用矩阵指数来找到状态转移矩阵：

$$\Phi(t) = e^{\mathbf At} = \mathbf{I} + \mathbf At  + \frac{(\mathbf At)^2}{2!} + \frac{(\mathbf At)^3}{3!} + ... $$

前两个项给出

$$\mathbf F = \begin{bmatrix}1 & t \\ -(k/m) t & 1-(c/m) t \end{bmatrix}$$

这可能足够精确，但也可能不够。您可以通过计算您的常数的 $\frac{(\mathbf At)^2}{2!}$ 并查看此矩阵对结果的贡献来轻松检查此内容。

### 线性时不变理论

[*线性时不变理论*](https://en.wikipedia.org/wiki/LTI_system_theory)，也称为 LTI 系统理论，可以通过使用逆拉普拉斯变换来找到 $\Phi$。您现在可能点头示意，或完全迷失。我将不会在本书中使用拉普拉斯变换。LTI 系统理论告诉我们

$$ \Phi(t) = \mathcal{L}^{-1}[(s\mathbf{I} - \mathbf{A})^{-1}]$$

我并不想深入探讨这个问题，只想说拉普拉斯变换$\mathcal{L}$将信号转换为不包含时间的空间$s$，但是要找到上述方程的解决方案并不容易。如果你感兴趣，LTI系统理论的维基百科文章提供了一个介绍。我提到LTI是因为你会发现有些文献使用它来设计卡尔曼滤波器矩阵以解决复杂的问题。

### 数值解法

最后，有一些数值技巧可以找到$\mathbf F$。随着滤波器变得越来越大，找到解析解变得非常繁琐（虽然像SymPy这样的软件包使其变得更加容易）。C. F. van Loan [2]开发了一种技术，可以通过数值方法计算出$\Phi$和$\mathbf Q$。给出连续模型

$$ \dot x = Ax + Gw$$

其中$w$是单位白噪声，van Loan的方法计算出$\mathbf F_k$和$\mathbf Q_k$。

我在`FilterPy`中实现了van Loan的方法。你可以按照以下步骤使用它：

In [ ]:
from filterpy.common import van_loan_discretization

A = np.array([[0., 1.], [-1., 0.]])
G = np.array([[0.], [2.]]) # 白noise比例
F, Q = van_loan_discretization(A, G, dt=0.1)

在 *微分方程的数值积分* 章节中，我提供了卡尔曼滤波中常用的替代方法。

## 过程噪声矩阵的设计

一般而言，$\mathbf Q$ 矩阵的设计是卡尔曼滤波器设计中最困难的方面之一。这是由于几个因素造成的。首先，数学需要对信号理论有良好的基础。第二，我们试图建模我们很少有信息的东西中的噪声。考虑尝试为一个投掷的棒球建模过程噪声。我们可以将其建模为一个在空气中移动的球体，但这留下了许多未知因素 - 球的旋转和旋转衰变，带缝合的球的阻力系数，风的影响和空气密度等等。我们为给定过程模型开发了精确数学解的方程，但由于过程模型不完整，$\mathbf Q$ 的结果也将是不完整的。这对卡尔曼滤波器的行为有很多影响。如果 $\mathbf Q$ 太小，则滤波器将对其预测模型过于自信，并会与实际解偏离。如果 $\mathbf Q$ 太大，那么...

滤波器会受到测量噪声的不合理影响，表现不佳。在实践中，我们会花费大量时间运行模拟和评估收集到的数据，以尝试选择一个适当的 $\mathbf Q$ 值。但让我们先看看数学公式。

假设一个运动学系统——一个可以用牛顿运动方程建模的系统。我们可以对这个过程做出几种不同的假设。

我们一直在使用一个过程模型：

$$ \dot{\mathbf x} = \mathbf{Ax} + \mathbf{Bu} + \mathbf{w}$$

其中 $\mathbf{w}$ 是过程噪声。运动学系统是*连续*的——它们的输入和输出可以在任意时间点变化。然而，我们的卡尔曼滤波器是*离散*的（卡尔曼滤波器有连续形式，但我们不在本书中涵盖）。我们以固定的时间间隔对系统进行采样。因此，我们必须找到上面方程中噪声项的离散表示。这取决于我们对噪声行为做出的假设。我们将考虑两种不同的噪声模型。

### 连续白噪声模型

我们使用牛顿方程来建模运动系统。我们通常使用位置和速度，或者位置、速度和加速度作为我们系统的模型。没有什么能阻止我们进一步建模——我们可以建模加加速度、加加加速度等等。通常我们不这样做，因为超出真实系统动力学的项会降低估计精度。

假设我们需要对位置、速度和加速度进行建模。那么我们可以假设每个离散时间步长中加速度是恒定的。当然，系统中存在过程噪声，因此加速度实际上并非恒定的。由于外部未建模的力，被跟踪的物体会随时间改变加速度。在本节中，我们假设加速度通过连续时间零均值白噪声$w(t)$变化。换句话说，我们假设速度的微小变化随时间平均值为0（零均值）。

由于噪声在不断变化，我们需要进行积分以获取我们选择的离散化间隔的离散噪声。我们不会在这里证明它，但是离散化噪声的方程式为

$$\mathbf Q = \int_0^{\Delta t} \mathbf F(t)\mathbf{Q_c}\mathbf F^\mathsf{T}(t) dt$$

其中$\mathbf{Q_c}$是连续噪声。一般的推理应该是清楚的。$\mathbf F(t)\mathbf{Q_c}\mathbf F^\mathsf{T}(t)$是基于我们的过程模型$\mathbf F(t)$在时刻$t$进行的连续噪声的投影。我们想知道在一个离散的间隔$\Delta t$内系统添加了多少噪声，因此我们在区间$[0,\Delta t]$上积分这个表达式。

我们知道牛顿系统的基本矩阵为

$$F = \begin{bmatrix}1 & \Delta t & {\Delta t}^2/2 \\ 0 & 1 & \Delta t\\ 0& 0& 1\end{bmatrix}$$

我们将连续噪声定义为

$$\mathbf{Q_c} = \begin{bmatrix}0&0&0\\0&0&0\\0&0&1\end{bmatrix} \Phi_s$$

其中$\Phi_s$是白噪声的频谱密度。这可以推导出来，但超出了本书的范围。有关详细信息，请参阅任何一本关于随机过程的标准文本。在实践中，我们通常不知道噪声的频谱密度，因此这就成为一个“工程”因素-一个我们通过实验调整的数字，直到我们的滤波器按我们的期望执行为止。您可以看到，$\Phi_s$相乘的矩阵实际上将功率谱密度分配给加速度项。这是有道理的；我们假设系统除了由噪声引起的变化外，具有恒定加速度。噪声改变了加速度。

我们可以自己进行这些计算，但我更喜欢使用SymPy解决方程。

$$\mathbf{Q_c} = \begin{bmatrix}0&0&0\\0&0&0\\0&0&1\end{bmatrix} \Phi_s$$

In [ ]:
import sympy
from sympy import (init_printing, Matrix, MatMul, 
                   integrate, symbols)

In [ ]:
init_printing(use_latex='mathjax')
dt, phi = symbols('\Delta{t} \Phi_s')
F_k = Matrix([[1, dt, dt**2/2],
              [0,  1,      dt],
              [0,  0,       1]])
Q_c = Matrix([[0, 0, 0],
              [0, 0, 0],
              [0, 0, 1]])*phi

Q = integrate(F_k * Q_c * F_k.T, (dt, 0, dt))

# 将矩阵中的phi提取出来，以使之更易读懂
Q = Q / phi
MatMul(Q, phi)

为了完整起见，我们计算了0阶和1阶方程的方程式。

In [ ]:
F_k = Matrix([[1]])
Q_c = Matrix([[phi]])

print('0阶离散过程noise')
integrate(F_k*Q_c*F_k.T,(dt, 0, dt))

In [ ]:
F_k = Matrix([[1, dt],
              [0, 1]])
Q_c = Matrix([[0, 0],
              [0, 1]]) * phi

Q = integrate(F_k * Q_c * F_k.T, (dt, 0, dt))

print('1阶离散过程noise')
# 将矩阵中的phi提取出来，以使之更易读懂
Q = Q / phi
MatMul(Q, phi)

### 分段白噪声模型

另一个噪声模型假定最高阶项（例如加速度）在每个时间段内是恒定的，但在每个时间段内略有不同，并且每个时间段之间相互不相关。换句话说，每个时间步长处都存在加速度的不连续跳跃。这与上述模型略有不同，上述模型假设最后一项有一个连续变化的噪声信号。

我们将其建模为

$$f(x)=Fx+\Gamma w$$

其中$\Gamma$是系统的*噪声增益*，$w$是常数分段加速度（或速度、抖动等）。

让我们首先看一个一阶系统。在这种情况下，我们有状态转移函数

$$\mathbf{F} = \begin{bmatrix}1&\Delta t \\ 0& 1\end{bmatrix}$$

在一个时间段内，速度的变化量将是$w(t)\Delta t$，位置的变化量将是$w(t)\Delta t^2/2$，因此我们得到

$$\Gamma = \begin{bmatrix}\frac{1}{2}\Delta t^2 \\ \Delta t\end{bmatrix}$$

过程噪声的协方差为

$$Q = \mathbb E[\Gamma w(t) w(t) \Gamma^\mathsf{T}] = \Gamma\sigma^2_v\Gamma^\mathsf{T}$$.

我们可以使用SymPy计算如下

In [ ]:
var = symbols('sigma^2_v')
v = Matrix([[dt**2 / 2], [dt]])

Q = v * var * v.T

# factor variance out of the matrix to make it more readable
Q = Q / var
MatMul(Q, var)

二阶系统的运算方式相同。

$$\mathbf{F} = \begin{bmatrix}1 & \Delta t & {\Delta t}^2/2 \\ 0 & 1 & \Delta t\\ 0& 0& 1\end{bmatrix}$$

在这里，我们假设白噪声是离散时间维纳过程。这给了我们

$$\Gamma = \begin{bmatrix}\frac{1}{2}\Delta t^2 \\ \Delta t\\ 1\end{bmatrix}$$

这个模型没有“真实性”，只是方便并提供了良好的结果。例如，我们可以假设噪声施加在jerk上，代价是更复杂的方程式。

过程噪声的协方差为

$$Q = \mathbb E[\Gamma w(t) w(t) \Gamma^\mathsf{T}] = \Gamma\sigma^2_v\Gamma^\mathsf{T}$$。

我们可以使用 SymPy 计算如下：

In [ ]:
var = symbols('sigma^2_v')
v = Matrix([[dt**2 / 2], [dt], [1]])

Q = v * var * v.T

# 将方差从矩阵中分离出来，以使其更易读
Q = Q / var
MatMul(Q, var)

我们不能说这个模型比连续模型更正确或不正确——两者都是对实际物体发生的情况的近似。只有经验和实验才能指导您选择适当的模型。在实践中，您通常会发现两个模型都可以提供合理的结果，但一个模型通常会比另一个模型表现更好。

第二个模型的优点是我们可以用 $\sigma^2$ 表示噪声，并且可以根据我们期望的运动和误差量来描述它。第一个模型要求我们指定谱密度，这不是很直观，但它更容易处理不同的时间采样，因为噪声在时间段内被综合。但这些不是固定规则 - 根据滤波器的性能测试和/或您对物理模型行为的了解，使用任何模型（或您自己设计的模型）。

一个好的经验法则是将 $\sigma$ 设置在 $\frac{1}{2}\Delta a$ 到 $\Delta a$ 之间，其中 $\Delta a$ 是加速度在样本周期之间最大变化的量。在实践中，我们选择一个数字，在数据上运行模拟，并选择运行良好的值。

### 使用 FilterPy 计算 Q

FilterPy提供了几个例程来计算$\mathbf Q$矩阵。函数`Q_continuous_white_noise()`计算给定$\Delta t$和谱密度的$\mathbf Q$。

In [ ]:
from filterpy.common import Q_continuous_white_noise
from filterpy.common import Q_discrete_white_noise

Q = Q_continuous_white_noise(dim=2, dt=1, spectral_density=1)
print(Q)

In [ ]:
Q = Q_continuous_white_noise(dim=3, dt=1, spectral_density=1)
print(Q)

函数`Q_discrete_white_noise()`假设噪声模型是分段的来计算$\mathbf Q$。

In [ ]:
Q = Q_discrete_white_noise(2, var=1.)
print(Q)

In [ ]:
Q = Q_discrete_white_noise(3, var=1.)
print(Q)

### $\mathbf Q$的简化

许多处理方法使用一个更简单的$\mathbf Q$形式，除了右下角的噪声项外，将其设为零。这样做合理吗？考虑小$\Delta t$时$\mathbf Q$的值。

In [ ]:
import numpy as np

In [ ]:
np.set_printoptions(precision=8)
Q = Q_continuous_white_noise(
    dim=3, dt=0.05, spectral_density=1)
print(Q)
np.set_printoptions(precision=3)

我们可以看到大多数项都非常小。请记住，使用此矩阵的唯一方程式是

$$ \mathbf P=\mathbf{FPF}^\mathsf{T} + \mathbf Q$$

如果相对于 $\mathbf P$，$\mathbf Q$ 的值很小，则几乎不会对 $\mathbf P$ 的计算产生任何贡献。将 $\mathbf Q$ 设置为除右下项外的零矩阵

$$\mathbf Q=\begin{bmatrix}0&0&0\\0&0&0\\0&0&\sigma^2\end{bmatrix}$$

虽然不正确，但通常是一个有用的近似。如果您将此应用于重要应用程序，则必须执行多次研究以确保您的滤波器在各种情况下都能正常工作。

如果这样做，“右下角项”表示每个变量的变化最快的项。如果状态为$x=\begin{bmatrix}x & \dot x & \ddot{x} & y & \dot{y} & \ddot{y}\end{bmatrix}^\mathsf{T}$，那么$\mathbf Q$将是6x6的；在$\mathbf Q$中，$\ddot{x}$和$\ddot{y}$的元素必须设置为非零。

## 后验协方差的稳定计算

我已经提出了计算后验协方差的方程式

$$\mathbf P = (\mathbf I - \mathbf{KH})\mathbf{\bar P}$$

严格来说，这是正确的，但这不是我在`FilterPy`中使用的计算方法，我使用*Joseph*方程式

$$\mathbf P = (\mathbf I-\mathbf {KH})\mathbf{\bar P}(\mathbf I-\mathbf{KH})^\mathsf T + \mathbf{KRK}^\mathsf T$$

我经常收到电子邮件和/或GitHub问题，声称实现有bug。它不是bug，我有几个原因使用它。首先，减法$(\mathbf I - \mathbf{KH})$可能会因为浮点误差导致非对称矩阵结果。 协方差必须对称，因此变得非对称通常会导致卡尔曼滤波器发散，甚至导致代码引发异常，因为`NumPy`内置了检查。

保持对称的传统方法是以下公式：

$$\mathbf P = (\mathbf P + \mathbf P^\mathsf T) / 2$$

这是安全的，因为矩阵中所有协方差的$\sigma_{ij} = \sigma_{ji}$。因此，如果它们因为浮点误差而发散，此操作将平均两个值之间的差异误差。

如果您查看上述方程的Joseph形式，您将会看到两个项中都有类似的$\mathbf{ABA}^\mathsf T$模式。因此，它们都保持对称。但是这个方程从哪里来，为什么我使用它而不是

$$\mathbf P = (\mathbf I - \mathbf{KH})\mathbf{\bar P} \\
\mathbf P = (\mathbf P + \mathbf P^\mathsf T) / 2$$

让我们从第一原理推导这个方程。这并不太难，您需要理解这个方程的目的，更重要的是，如果您的滤波器因数值不稳定性而发散，需要诊断问题。这个推导来自于Brown[4]。

首先，一些符号。$\mathbf x$是我们系统的真实状态。$\mathbf{\hat x}$是我们系统的估计状态，即后验状态。$\mathbf{\bar x}$是系统的先验估计。


鉴于此，我们可以定义我们的模型为

$$\mathbf x_{k+1} = \mathbf F_k \mathbf x_k + \mathbf w_k \\
\mathbf z_k = \mathbf H_k \mathbf x_k + \mathbf v_k$$

用文字表述，系统的下一状态 $\mathbf x_{k+1}$ 是当前状态 $k$ 加上某个过程 $\mathbf F_k$ 的变化再加上一些噪声 $\mathbf w_k$。

需要注意的是这些只是定义。没有一个系统是完全按照数学模型运行的，因此我们用噪声项 $\mathbf w_k$ 来建模。由于测量存在传感器误差，因此我们用 $\mathbf v_k$ 来建模。

在剩余的推导中，我们将只考虑步骤 $k$ 的值，而不是步骤 $k+1$。因此我们不再使用下标 $k$。

现在我们定义估计误差为真实状态和估计状态之间的差异：

$$ \mathbf e = \mathbf x - \mathbf{\hat x}$$

同样，这只是一种定义；我们不知道如何计算 $\mathbf e$，它只是真实状态和估计状态之间的定义差异。

这使我们能够定义我们估计的协方差，它被定义为 $\mathbf{ee}^\mathsf T$ 的期望值：

$$\begin{aligned}
P &= E[\mathbf{ee}^\mathsf T] \\
&= E[(\mathbf x - \mathbf{\hat x})(\mathbf x - \mathbf{\hat x})^\mathsf T]
\end{aligned}$$


接下来，我们定义后验估计为

$$\mathbf {\hat x} = \mathbf{\bar x} + \mathbf K(\mathbf z - \mathbf{H \bar x})$$

这看起来像卡尔曼滤波器的方程，这是有原因的。但是和到目前为止的所有数学一样，这是一个**定义**。特别地，我们没有定义$\mathbf K$，你不应该认为它是卡尔曼增益，因为我们解决任何问题，而不仅仅是线性卡尔曼滤波器。在这里，$\mathbf K$只是介于0和1之间的一些未指定的混合值。

现在我们有了我们的定义，让我们进行一些替换和代数运算。

该项$(\mathbf x - \mathbf{\hat x})$可以通过将$\mathbf{\hat x}$替换为上述定义来扩展，从而得到

$$(\mathbf x - \mathbf{\hat x}) = \mathbf x - (\mathbf{\bar x} + \mathbf K(\mathbf z - \mathbf{H \bar x}))$$

现在我们将$\mathbf z$替换为$\mathbf H \mathbf x + \mathbf v$:

$$\begin{aligned}
(\mathbf x - \mathbf{\hat x})
&= \mathbf x - (\mathbf{\bar x} + \mathbf K(\mathbf z - \mathbf{H \bar x})) \\
&= \mathbf x - (\mathbf{\bar x} + \mathbf K(\mathbf H \mathbf x + \mathbf v - \mathbf{H \bar x})) \\
&= (\mathbf x - \mathbf{\bar x}) - \mathbf K(\mathbf H \mathbf x + \mathbf v - \mathbf{H \bar x}) \\
&= (\mathbf x - \mathbf{\bar x}) - \mathbf{KH}(\mathbf x - \mathbf{ \bar x}) - \mathbf{Kv} \\
&=  (\mathbf I - \mathbf{KH})(\mathbf x - \mathbf{\bar x}) - \mathbf{Kv}
\end{aligned}$$

现在，如果我们注意到$(\mathbf x - \mathbf{\bar x})$的期望值是先验协方差$\mathbf{\bar P}$，而$\mathbf v$的期望值是$E[\mathbf{vv}^\mathbf T] = \mathbf R$，我们就可以解出$\mathbf P$：

$$\begin{aligned}
\mathbf P &= 
   E\big[[(\mathbf I - \mathbf{KH})(\mathbf x - \mathbf{\bar x}) - \mathbf{Kv})]
  [(\mathbf I - \mathbf{KH})(\mathbf x - \mathbf{\bar x}) - \mathbf{Kv}]^\mathsf T\big ] \\
  &= (\mathbf I - \mathbf{KH})\mathbf{\bar P}(\mathbf I - \mathbf{KH})^\mathsf T + \mathbf{KRK}^\mathsf T
\end{aligned}$$

这就是我们要证明的内容。

请注意，这个等式对于*任何* $\mathbf K$ 都是有效的，不仅适用于由 Kalman 滤波器计算出的最佳 $\mathbf K$。这就是我使用这个等式的原因。在实践中，由滤波器计算出的 Kalman 增益*不是*最优值，因为现实世界从未真正呈线性和高斯分布，而且计算中由于浮点误差而产生。在面对实际情况时，这个等式远不可能导致 Kalman 滤波器发散。

那么$\mathbf P = (\mathbf I - \mathbf{KH})\mathbf{\bar P}$是从哪里来的呢？让我们完成这个简单的推导。回想一下，Kalman滤波器（最优）增益由以下公式给出：

$$\mathbf K = \mathbf{\bar P H^\mathsf T}(\mathbf{H \bar P H}^\mathsf T + \mathbf R)^{-1}$$

现在我们将其代入刚刚推导出的方程：

$$\begin{aligned}
&= (\mathbf I - \mathbf{KH})\mathbf{\bar P}(\mathbf I - \mathbf{KH})^\mathsf T + \mathbf{KRK}^\mathsf T\\
&= \mathbf{\bar P} - \mathbf{KH}\mathbf{\bar P} - \mathbf{\bar PH}^\mathsf T\mathbf{K}^\mathsf T + \mathbf K(\mathbf{H \bar P H}^\mathsf T + \mathbf R)\mathbf K^\mathsf T \\
&= \mathbf{\bar P} - \mathbf{KH}\mathbf{\bar P} - \mathbf{\bar PH}^\mathsf T\mathbf{K}^\mathsf T + \mathbf{\bar P H^\mathsf T}(\mathbf{H \bar P H}^\mathsf T + \mathbf R)^{-1}(\mathbf{H \bar P H}^\mathsf T + \mathbf R)\mathbf K^\mathsf T\\
&= \mathbf{\bar P} - \mathbf{KH}\mathbf{\bar P} - \mathbf{\bar PH}^\mathsf T\mathbf{K}^\mathsf T + \mathbf{\bar P H^\mathsf T}\mathbf K^\mathsf T\\
&= \mathbf{\bar P} - \mathbf{KH}\mathbf{\bar P}\\
&= (\mathbf I - \mathbf{KH})\mathbf{\bar P}
\end{aligned}$$

因此当增益最优时，$\mathbf P = (\mathbf I - \mathbf{KH})\mathbf{\bar P}$ 在数学上是正确的，但是$(\mathbf I - \mathbf{KH})\mathbf{\bar P}(\mathbf I - \mathbf{KH})^\mathsf T + \mathbf{KRK}^\mathsf T$也是正确的，当增益不是最优时，它也更加数值稳定。因此，我在 FilterPy 中使用这个计算方法。

你的滤波器仍然可能发散，特别是如果它运行了数百或数千个时期。你需要检查这些方程式。文献提供了其他形式的计算，可能更适用于你的问题。和往常一样，如果你正在解决真正的工程问题，其中故障可能意味着设备或生命的损失，你需要超越本书，阅读工程文献。如果你正在处理无害的“玩具”问题，如果检测到发散，可以将$\mathbf P$的值重置为一些“合理”的值，然后继续。例如，你可以将非对角线元素归零，以便矩阵只包含方差，然后乘以某个略大于1的常数，以反映你刚刚注入到滤波器中的信息损失。发挥你的想象力并进行测试。

## 推导卡尔曼增益方程

如果你已经读完了上一节，那么也可以读一下这一节。通过这一节，我们将得出卡尔曼滤波器方程式。

请注意，这个推导过程*没有*使用贝叶斯公式。我见过至少四种不同的卡尔曼滤波器方程式推导方法；这个推导过程是文献中典型的方法，符合上一节的内容。参考文献再次是Brown [4]。

在上一节中，我们使用了一个未指定的缩放因子$\mathbf K$来推导协方差方程的Joseph形式。如果我们想要一个最佳滤波器，我们需要使用微积分来最小化方程中的误差。你应该对这个想法很熟悉。如果你想找到函数$f(x)$的最小值，你需要求导并将其置为零：$\frac{x}{dx}f(x) = 0$。

在我们的问题中，误差由协方差矩阵 $\mathbf P$ 表示。特别地，对角线表示状态向量中每个元素的误差（方差）。因此，要找到最佳增益，我们希望对对角线的迹（和）进行求导。

Brown 提醒我们有两个涉及迹的导数公式：

$$\frac{d\, trace(\mathbf{AB})}{d\mathbf A} = \mathbf B^\mathsf T$$

$$\frac{d\, trace(\mathbf{ACA}^\mathsf T)}{d\mathbf A} = 2\mathbf{AC}$$

其中，$\mathbf{AB}$ 是方阵，$\mathbf C$ 是对称的。

我们将 Joseph 方程展开为：

$$\mathbf P = \mathbf{\bar P} - \mathbf{KH}\mathbf{\bar P} - \mathbf{\bar P}\mathbf H^\mathsf T \mathbf K^\mathsf T + \mathbf K(\mathbf H \mathbf{\bar P}\mathbf H^\mathsf T + \mathbf R)\mathbf K^\mathsf T$$

现在，我们需要求关于 $\mathbf K$ 的 $\mathbf P$ 迹的导数：$\frac{d\, trace(\mathbf P)}{d\mathbf K}$。

第一项对 $\mathbf K$ 的迹的导数为 $0$，因为它的表达式中没有 $\mathbf K$。

第二项的迹的导数为 $(\mathbf H\mathbf{\bar P})^\mathsf T$。

我们可以通过注意到 $\mathbf{\bar P}\mathbf H^\mathsf T \mathbf K^\mathsf T$ 是 $\mathbf{KH}\mathbf{\bar P}$ 的转置来找到第三项的迹的导数。矩阵的迹等于它的转置的迹，所以它的导数与第二项相同。

最后，第四项的迹的导数是 $2\mathbf K(\mathbf H \mathbf{\bar P}\mathbf H^\mathsf T + \mathbf R)$。

这给我们最终的值：

$$\frac{d\, trace(\mathbf P)}{d\mathbf K} = -2(\mathbf H\mathbf{\bar P})^\mathsf T + 2\mathbf K(\mathbf H \mathbf{\bar P}\mathbf H^\mathsf T + \mathbf R)$$

我们将其设置为零并解出 $\mathbf K$ 的方程，以最小化误差。

$$-2(\mathbf H\mathbf{\bar P})^\mathsf T + 2\mathbf K(\mathbf H \mathbf{\bar P}\mathbf H^\mathsf T + \mathbf R) = 0 \\
\mathbf K(\mathbf H \mathbf{\bar P}\mathbf H^\mathsf T + \mathbf R) = (\mathbf H\mathbf{\bar P})^\mathsf T \\
\mathbf K(\mathbf H \mathbf{\bar P}\mathbf H^\mathsf T + \mathbf R) = \mathbf{\bar P}\mathbf H^\mathsf T \\
\mathbf K= \mathbf{\bar P}\mathbf H^\mathsf T (\mathbf H \mathbf{\bar P}\mathbf H^\mathsf T + \mathbf R)^{-1}
$$

这个推导并不完全严谨，因为我省略了为什么最小化迹会最小化总误差的论证，但我认为对这本书来说已经足够了。如果您需要更详细的内容，任何标准教材都会有更深入的阐述。

## 微分方程的数值积分

我们已经接触了几种解决线性微分方程的数值技巧。其中包括状态空间方法、拉普拉斯变换和范洛恩的方法。

这些方法对于线性常微分方程（ODE）效果很好，但对于非线性方程效果不佳。例如，考虑尝试预测快速转弯汽车的位置。汽车通过转动前轮进行机动。这使得它们围绕其向前移动的后轴旋转。因此，路径将不断变化，线性预测将必然产生错误的值。如果相对于 $\Delta t$ 系统的变化足够小，这通常可以产生足够的结果，但在我们将在后续章节中研究的非线性卡尔曼滤波器中很少会出现这种情况。

因此，出于这些原因，我们需要知道如何数值积分 ODE。这可以是一个需要几本书的广泛主题。但是，我将介绍一些简单的技术，这些技术对于您遇到的大多数问题都有效。

### 欧拉法

假设我们有以下初值问题：

$$\begin{gathered}
y' = y, \\ y(0) = 1
\end{gathered}$$

我们碰巧知道这个方程的确切解为 $y=e^t$ ，因为我们之前已经解决过，但对于任意的常微分方程，我们都不会知道确切的解。我们通常只知道方程的导数，这相当于斜率。我们还知道初始值：在 $t=0$ 时，$y=1$。如果我们知道这两个信息，我们就可以使用 $t=0$ 处的斜率和 $y(0)$ 的值来预测 $y(t=1)$ 的值。我在下面绘制了这个图。

In [ ]:
import matplotlib.pyplot as plt

t = np.linspace(-1, 1, 10)
plt.plot(t, np.exp(t))
t = np.linspace(-1, 1, 2)
plt.plot(t,t+1, ls='--', c='k');

你可以看到，当$t=0.1$时，斜率非常接近曲线，但是当$t=1$时，相距甚远。但是让我们暂时继续使用步长为1。我们可以看到，在$t=1$时，$y$的估计值为2。现在我们可以通过取$t=1$时曲线的斜率并将其加到我们的初始估计值中来计算$t=2$时的值。斜率通过$y'=y$计算，因此斜率为2。

In [ ]:
import kf_book.book_plots as book_plots

t = np.linspace(-1, 2, 20)
plt.plot(t, np.exp(t))
t = np.linspace(0, 1, 2)
plt.plot([1, 2, 4], ls='--', c='k')
book_plots.set_labels(x='x', y='y');

在这里，我们看到$y$的下一个估计值为4。误差很快就变大了，你可能并不满意。但是1是一个非常大的步长。让我们将这个算法放入代码中，并通过使用小的步长来验证它是否有效。

In [ ]:
def euler(t, tmax, y, dx, step=1.):
    ys = []
    while t < tmax:
        y = y + step*dx(t, y)
        ys.append(y)
        t +=step        
    return ys

In [ ]:
def dx(t, y): return y

print(euler(0, 1, 1, dx, step=1.)[-1])
print(euler(0, 2, 1, dx, step=1.)[-1])

这看起来正确。现在让我们绘制一个更小步长的结果。

In [ ]:
ys = euler(0, 4, 1, dx, step=0.00001)
plt.subplot(1,2,1)
plt.title('计算的')
plt.plot(np.linspace(0, 4, len(ys)),ys)
plt.subplot(1,2,2)
t = np.linspace(0, 4, 20)
plt.title('精确的')
plt.plot(t, np.exp(t));

In [ ]:
print('精确答案=', np.exp(4))
print('欧拉法答案=', ys[-1])
print('差=', np.exp(4) - ys[-1])
print('迭代次数=', len(ys))

在这里，我们看到误差是相当小的，但要获得三位数的精度需要很多迭代次数。在实践中，欧拉法对于大多数问题来说速度太慢，我们使用更复杂的方法。

在继续之前，让我们正式推导欧拉方法，因为它是下一节中使用的更高级的龙格-库塔方法的基础。实际上，欧拉方法是龙格-库塔方法的最简单形式。

以下是 $y$ 的泰勒展开的前 3 项。无限展开将给出一个精确答案，因此 $O(h^4)$ 表示由于有限展开而产生的误差。

$$y(t_0 + h) = y(t_0) + h y'(t_0) + \frac{1}{2!}h^2 y''(t_0) + \frac{1}{3!}h^3 y'''(t_0) +  O(h^4)$$

在这里，我们可以看到欧拉方法使用了泰勒展开的前两项。每个后续项都比前面的项小，因此我们可以确信估计值不会远离正确值。

### 龙格-库塔方法

Runge Kutta是数值积分的工作马。文献中有大量的方法。实践中，使用我在这里介绍的Runge Kutta算法将解决你所面临的大多数问题。它提供了非常好的速度、精度和稳定性平衡，是“默认”的数值积分方法，除非你有非常好的理由选择其他方法。

让我们深入探讨。我们从一些微分方程开始

$$\ddot{y} = \frac{d}{dt}\dot{y}$$。

我们可以用一个函数f代替y的导数，如下所示

$$\ddot{y} = \frac{d}{dt}f(y,t)$$。

导出这些方程不在本书的范围之内，但Runge Kutta RK4方法是用这些方程定义的。

$$y(t+\Delta t) = y(t) + \frac{1}{6}(k_1 + 2k_2 + 2k_3 + k_4) + O(\Delta t^4)$$

$$\begin{aligned}
k_1 &= f(y,t)\Delta t \\
k_2 &= f(y+\frac{1}{2}k_1, t+\frac{1}{2}\Delta t)\Delta t \\
k_3 &= f(y+\frac{1}{2}k_2, t+\frac{1}{2}\Delta t)\Delta t \\
k_4 &= f(y+k_3, t+\Delta t)\Delta t
\end{aligned}
$$

以下是相应的代码：

In [ ]:
def runge_kutta4(y, x, dx, f):
    """为dy/dx计算第4阶Runge-Kutta方法。
    y是y的初始值
    x是x的初始值
    dx是x的差异（例如时间步骤）
    f是可调用函数（y，x），您可以提供该函数以计算指定值的dy / dx。
    """
    
    k1 = dx * f(y, x)
    k2 = dx * f(y + 0.5*k1, x + 0.5*dx)
    k3 = dx * f(y + 0.5*k2, x + 0.5*dx)
    k4 = dx * f(y + k3, x + dx)
    
    return y + (k1 + 2*k2 + 2*k3 + k4) / 6.

让我们用一个简单的例子来使用它。

$$\dot{y} = t\sqrt{y(t)}$$

初始值为

$$\begin{aligned}t_0 &= 0\\y_0 &= y(t_0) = 1\end{aligned}$$

In [ ]:
import math
import numpy as np
t = 0.
y = 1.
dt = .1

ys, ts = [], []

In [ ]:
def func(y, t):
    return t * math.sqrt(y)

while t <= 10:
    y = runge_kutta4(y, t, dt, func)
    t += dt
    ys.append(y)
    ts.append(t)

exact = [(t ** 2 + 4) ** 2 / 16. for t in ts]
plt.plot(ts, ys)
plt.plot(ts, exact)

error = np.array(exact) - np.array(ys)
print(f"最大误差 {max(error):.5f}")

## 贝叶斯滤波

从离散贝叶斯章节开始，我们使用贝叶斯公式进行滤波。假设我们正在跟踪一个物体，将其在特定时间的状态定义为其位置、速度等。例如，我们可以将时间 $t$ 的状态写成 $\mathbf x_t = \begin{bmatrix}x_t &\dot x_t \end{bmatrix}^\mathsf T$。

当我们对物体进行测量时，我们测量的是状态或其一部分。传感器存在噪声，因此测量值存在噪声。但是，测量结果显然由状态确定。也就是说，状态的变化可能会改变测量结果，但是测量结果的变化不会改变状态。

在滤波中，我们的目标是从时间0到时间$t$计算一组状态$\mathbf x_{0:t}$的最优估计值。如果我们知道$\mathbf x_{0:t}$，那么计算与这些状态对应的一组测量$\mathbf z_{0:t}$将是微不足道的。然而，我们接收到一组测量$\mathbf z_{0:t}$，并希望计算相应的状态$\mathbf x_{0:t}$。这被称为*统计反演*，因为我们试图从输出中计算输入。

反演是一个困难的问题，因为通常没有唯一的解。对于给定的状态$\mathbf x_{0:t}$，只有一组可能的测量值（加上噪声），但对于给定的一组测量值，有许多不同的状态集合可能导致这些测量值。

回想一下贝叶斯定理：

$$P(x \mid z) = \frac{P(z \mid x)P(x)}{P(z)}$$

其中$P(z \mid x)$是测量$z$的似然，$P(x)$是基于我们的过程模型的先验，$P(z)$是一个归一化常数。$P(x \mid z)$是后验或整合测量$z$后的分布，也称为证据。

这是一种*统计反演*，因为它从$P(z \mid x)$转变为$P(x \mid z)$。我们过滤问题的解决方案可以表示为：

$$P(\mathbf x_{0:t} \mid \mathbf z_{0:t}) = \frac{P(\mathbf z_{0:t} \mid \mathbf x_{0:t})P(\mathbf x_{0:t})}{P(\mathbf z_{0:t})}$$

这样就可以解决所有问题，直到下一个测量$\mathbf z_{t+1}$到来，此时我们需要重新计算区间$0:t+1$的整个表达式。

实际上，这是不可行的，因为我们试图计算在整个时间步长范围内状态的后验分布$P(\mathbf x_{0:t} \mid \mathbf z_{0:t})$。但是我们真的关心在第三个步骤（比如说）的概率分布吗，当我们刚刚收到第十个测量时？通常不关心。所以我们放松要求，只计算当前时间步长的分布。

第一个简化是我们将我们的过程（例如，移动物体的运动模型）描述为一个*马尔可夫链*。也就是说，我们说当前状态仅依赖于上一个状态和一个转移概率$P(\mathbf x_k \mid \mathbf x_{k-1})$，这只是从上一个状态到当前状态的概率。我们写成：

$$\mathbf x_k \sim P(\mathbf x_k \mid \mathbf x_{k-1})$$

实际上，这是非常合理的，因为许多事物具有*马尔可夫属性*。如果您正驾驶在停车场，您在下一秒的位置是否取决于您一分钟前是否从州际公路上下来或者在泥路上缓慢行驶？不是的。您下一秒的位置仅取决于当前位置、速度和控制输入，而不取决于一分钟前发生了什么。因此，汽车具有马尔可夫属性，我们可以进行这种简化而不会损失精度或一般性。

我们进行的下一个简化是将*测量模型*定义为取决于当前状态$\mathbf x_k$，给定当前状态的条件概率的测量值：$P(\mathbf z_k \mid \mathbf x_k)$。我们写成：

$$\mathbf z_k \sim P(\mathbf z_k \mid \mathbf x_k)$$

现在我们有了一个递归，因此我们需要一个初始条件来终止它。因此，我们说初始分布是状态$\mathbf x_0$的概率：

$$\mathbf x_0 \sim P(\mathbf x_0)$$


这些术语被插入到贝叶斯方程中。如果我们已知状态 $\mathbf x_0$ 和第一次测量，我们就可以估计 $P(\mathbf x_1 | \mathbf z_1)$。运动模型创建先验分布 $P(\mathbf x_2 \mid \mathbf x_1)$。我们将其反馈到贝叶斯定理中以计算 $P(\mathbf x_2 | \mathbf z_2)$。我们继续这个预测-校正算法，递归地计算时间 $t$ 的状态和分布，仅基于时间 $t-1$ 的状态和分布以及时间 $t$ 的测量。

这个计算的数学细节因问题而异。**离散贝叶斯**和**单变量卡尔曼滤波器**两章介绍了两种不同的公式，你应该可以理解。单变量卡尔曼滤波器假设对于标量状态，噪声和过程都是线性模型，受到均值为零、不相关高斯噪声的影响。

多元卡尔曼滤波器做出了相同的假设，但是对于向量而不是标量的状态和测量。卡尔曼博士能够证明，如果这些假设成立，那么卡尔曼滤波器从最小二乘意义上来说是*最优的*。俗话说，这意味着没有办法从嘈杂的测量中推导出更多的信息。在本书的剩余部分中，我将介绍放松线性和高斯噪声约束的滤波器。

在我继续之前，再谈一些统计反演的内容。正如Calvetti和Somersalo在《Bayesian Scientific Computing简介》中所写的那样，“我们采用贝叶斯观点：*随机性仅仅意味着缺乏信息*”[3]。我们的状态参数化了我们原则上可以测量或计算的物理现象：速度、空气阻力等等。我们缺乏足够的信息来计算或测量它们的值，因此我们选择将它们视为随机变量。严格来说它们并不是随机的，因此这是一个主观的立场。

他们专门为这个主题撰写了一整章。我可以节省一段。贝叶斯滤波器是可能的，因为我们将统计属性归因于未知参数。在卡尔曼滤波器的情况下，我们有封闭形式的解来找到最优估计。其他滤波器，如我们在后面一章中介绍的离散贝叶斯滤波器或粒子滤波器，以更为临时、非最优的方式对概率进行建模。我们技术的力量来自于将缺乏信息视为随机变量，将该随机变量描述为概率分布，然后使用贝叶斯定理来解决统计推断问题。

## 将卡尔曼滤波器转换为g-h滤波器

我已经说过卡尔曼滤波器是g-h滤波器的一种形式。只需要进行一些代数证明即可。在一维情况下更为直观，因此我将在此进行。回想一下

$$
\mu_{x}=\frac{\sigma_1^2 \mu_2 + \sigma_2^2 \mu_1} {\sigma_1^2 + \sigma_2^2}
$$

我稍微修改了一下，以便更适合阅读：

$$
\mu_{x}=\frac{ya + xb} {a+b}
$$

我们可以通过以下代数运算很容易地将其转化为 g-h 形式：

$$
\begin{aligned}
\mu_{x}&=(x-x) + \frac{ya + xb} {a+b} \\
\mu_{x}&=x-\frac{a+b}{a+b}x  + \frac{ya + xb} {a+b} \\ 
\mu_{x}&=x +\frac{-x(a+b) + xb+ya}{a+b} \\
\mu_{x}&=x+ \frac{-xa+ya}{a+b}  \\
\mu_{x}&=x+ \frac{a}{a+b}(y-x)\\
\end{aligned}
$$

我们快要完成了，但是请记得估计的方差由以下公式给出：

$$\begin{aligned}
\sigma_{x}^2 &= \frac{1}{\frac{1}{\sigma_1^2} +  \frac{1}{\sigma_2^2}} \\
&= \frac{1}{\frac{1}{a} +  \frac{1}{b}}
\end{aligned}$$

我们可以通过观察以下事实将该项合并到上面的公式中：

$$ 
\begin{aligned}
\frac{a}{a+b} &= \frac{a/a}{(a+b)/a} = \frac{1}{(a+b)/a}  \\
 &= \frac{1}{1 + \frac{b}{a}} = \frac{1}{\frac{b}{b} + \frac{b}{a}}  \\
 &= \frac{1}{b}\frac{1}{\frac{1}{b} + \frac{1}{a}} \\
 &= \frac{\sigma^2_{x}}{b}
 \end{aligned}
$$

我们可以用下面的公式将所有内容联系起来：

$$
\begin{aligned}
\mu_{x}&=x+ \frac{a}{a+b}(y-x) \\
&= x + \frac{\sigma^2_{x}}{b}(y-x) \\
&= x + g_n(y-x)
\end{aligned}
$$

其中，

$$g_n = \frac{\sigma^2_{x}}{\sigma^2_{y}}$$

最终结果是将两次测量的残差乘以一个常数并加到先前的值上，这就是g-h滤波器的g方程。 $g$是新估计值的方差除以测量值的方差。当然，在这种情况下，$g$不是一个常数，因为方差会随着每个时间步长而变化。我们也可以以同样的方式推导出$h$的公式。这不是一个特别有启发性的推导，我会跳过它。最终结果是

$$h_n = \frac{COV (x,\dot x)}{\sigma^2_{y}}$$

重点是，在时间 $n$，通过测量和预测的方差和协方差完全确定了 $g$ 和 $h$。换句话说，我们通过一个由这两个输入质量决定的比例因子，在测量和预测之间选择一个点。

## 参考文献

 * [1] C.B. Molwer 和 C.F. Van Loan "Nineteen Dubious Ways to Compute the Exponential of a Matrix, Twenty-Five Years Later,", *SIAM Review 45, 3-49*. 2003.


 * [2] C.F. van Loan, "Computing Integrals Involving the Matrix Exponential," IEEE *Transactions Automatic Control*, June 1978.
 
 
 * [3] Calvetti, D 和 Somersalo E, "Introduction to Bayesian Scientific Computing: Ten Lectures on Subjective Computing,", *Springer*, 2007.
 
 * [4] Brown, R. G. 和 Hwang, P. Y.C.，"Introduction to Random Signals and Applied Kalman Filtering", *Wiley and Sons*, 第四版，第143-147页，2012。